# StoreListener: Bilingual Speech Recognition Pipeline

<a href="https://colab.research.google.com/github/MuhammadSarfrazAfzal/whisper-finetune/blob/main/storelistener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook implements the complete, modular parameter-efficient fine-tuning (PEFT / LoRA) and evaluation pipeline for **StoreListener** (Offline Automatic Speech Recognition for Retail).

### Objectives & Task Separation
* **Primary Training Objective (Phase 1 ASR)**:
  * English speech $\rightarrow$ English transcript (`task="transcribe"`, `<|en|><|transcribe|>`).
  * Spanish speech $\rightarrow$ Spanish transcript (`task="transcribe"`, `<|es|><|transcribe|>`).
* **Speech Translation (Phase 2)**: Spanish speech $\rightarrow$ English text (`task="translate"`, `<|es|><|translate|>`) is strictly separated and tested as an independent evaluation task.
* **Base Model**: `openai/whisper-base` (74M parameter multilingual checkpoint; `.en` checkpoints are excluded).
* **Dataset**: 600 verified clips (300 EN, 300 ES) from Google FLEURS; existing split assignments (488 train, 46 val, 66 test) are strictly preserved.
* **Safety Guard**: Default configuration is **READ-ONLY** (`EXECUTE_TRAINING = False`). Training requires explicit user opt-in.

In [ ]:
# Mount Google Drive when running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[+] Google Drive mounted at /content/drive")
except ImportError:
    print("[!] Note: google.colab not available. Running in local development environment.")


In [ ]:
import os

if os.path.exists('/content/drive/MyDrive'):
    print("[+] Google Drive contents in /content/drive/MyDrive:")
    print(os.listdir('/content/drive/MyDrive')[:10])  # Show first 10 items
else:
    print("[!] /content/drive/MyDrive not present (local execution).")


In [ ]:
import os

# Scan for dataset audio and manifests in Colab Google Drive or local workspace
root_dir = "/content/drive/MyDrive/whisper-finetune" if os.path.exists("/content/drive/MyDrive/whisper-finetune") else (
    "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "."
)

print(f"[*] Scanning directory: {root_dir}")
for root, dirs, files in os.walk(root_dir):
    dirs[:] = [d for d in dirs if d not in {
        ".trash", ".shortcut-targets-by-id", ".git", "__pycache__", ".ipynb_checkpoints"
    }]

    wav_count = sum(f.lower().endswith(".wav") for f in files)

    csv_files = [
        f for f in files
        if f.lower() in [
            "train.csv",
            "validation.csv",
            "test.csv",
            "dataset_manifest.csv"
        ]
    ]

    if wav_count or csv_files:
        print(f"  Folder: {root} -> WAV files: {wav_count}, Manifests: {csv_files}")


## 1. Environment & Minimal Dependency Setup
Installs required libraries without version conflicts and performs hardware diagnostics (CUDA, GPU model, VRAM, and mixed-precision strategy).

In [ ]:
# Install minimal compatible packages for Whisper fine-tuning and evaluation
!pip install -q soundfile datasets transformers peft accelerate evaluate jiwer

import sys
import torch
import soundfile as sf
import datasets
import transformers
import peft
import accelerate
import evaluate
import jiwer

print("=" * 60)
print("INSTALLED PACKAGE VERSIONS:")
print(f"  Python:       {sys.version.split()[0]}")
print(f"  PyTorch:      {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  Datasets:     {datasets.__version__}")
print(f"  PEFT:         {peft.__version__}")
print(f"  Accelerate:   {accelerate.__version__}")
print(f"  Evaluate:     {evaluate.__version__}")
print(f"  JiWER:        {jiwer.__version__}")
print(f"  SoundFile:    {sf.__version__}")
print("=" * 60)

# Hardware Diagnostics
CUDA_READY = torch.cuda.is_available()
USE_FP16 = False
USE_BF16 = False

print("\nHARDWARE DIAGNOSTICS:")
if CUDA_READY:
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    capability = torch.cuda.get_device_capability(0)
    USE_BF16 = torch.cuda.is_bf16_supported()
    USE_FP16 = not USE_BF16
    print(f"  [+] Active GPU:          {device_name}")
    print(f"  [+] Dedicated VRAM:      {vram_gb:.2f} GB")
    print(f"  [+] Compute Capability:  {capability[0]}.{capability[1]}")
    print(f"  [+] Precision Strategy:  {'BF16' if USE_BF16 else 'FP16'}")
else:
    print("  [!] WARNING: No CUDA GPU detected (running in CPU mode).")
    print("      In Google Colab, enable a GPU via: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU.")
    print("      Model training is blocked until a GPU accelerator is connected.")
print("=" * 60)


## 2. Configurable Local and Colab Dataset Paths
Configures the hybrid path resolver supporting both the Google Drive dataset root in Colab (`/content/drive/MyDrive/whisper-finetune`) and local paths for development (`D:\Finetunning\whisper-finetune` or relative `.`).

In [ ]:
from pathlib import Path
import pandas as pd

# 1. Configurable Dataset Root (Google Drive in Colab, Local for development)
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/whisper-finetune")
LOCAL_DEVELOPMENT_ROOT = Path(r"D:\Finetunning\whisper-finetune")
CWD_ROOT = Path(".").resolve()

# User override parameter (optional)
CUSTOM_DATASET_ROOT = None  # e.g., Path("/custom/path/whisper-finetune")

if CUSTOM_DATASET_ROOT is not None and Path(CUSTOM_DATASET_ROOT).exists():
    DATASET_ROOT = Path(CUSTOM_DATASET_ROOT).resolve()
    print(f"[+] Using user CUSTOM_DATASET_ROOT: {DATASET_ROOT}")
elif COLAB_DRIVE_ROOT.exists() and (COLAB_DRIVE_ROOT / "train.csv").exists():
    DATASET_ROOT = COLAB_DRIVE_ROOT
    print(f"[+] Using Google Drive Colab root: {DATASET_ROOT}")
elif LOCAL_DEVELOPMENT_ROOT.exists() and (LOCAL_DEVELOPMENT_ROOT / "train.csv").exists():
    DATASET_ROOT = LOCAL_DEVELOPMENT_ROOT
    print(f"[+] Using Local Development root: {DATASET_ROOT}")
elif (CWD_ROOT / "train.csv").exists():
    DATASET_ROOT = CWD_ROOT
    print(f"[+] Using Current Working Directory root: {DATASET_ROOT}")
else:
    raise FileNotFoundError(
        f"CRITICAL: train.csv not found in Colab Drive ({COLAB_DRIVE_ROOT}), "
        f"local development path ({LOCAL_DEVELOPMENT_ROOT}), or cwd ({CWD_ROOT})."
    )

# 2. Load existing CSV manifests without modifying them
train_df = pd.read_csv(DATASET_ROOT / "train.csv")
val_df   = pd.read_csv(DATASET_ROOT / "validation.csv")
test_df  = pd.read_csv(DATASET_ROOT / "test.csv")

print(f"\n[+] Loaded existing dataset manifests from {DATASET_ROOT}:")
print(f"    Train Split:      {len(train_df):3d} rows (EN: {sum(train_df['language']=='en')}, ES: {sum(train_df['language']=='es')})")
print(f"    Validation Split: {len(val_df):3d} rows (EN: {sum(val_df['language']=='en')}, ES: {sum(val_df['language']=='es')})")
print(f"    Test Split:       {len(test_df):3d} rows (EN: {sum(test_df['language']=='en')}, ES: {sum(test_df['language']=='es')})")
print(f"    Total Rows:       {len(train_df) + len(val_df) + len(test_df)} rows (Expected: 600)")

# Confirm split isolation (0 overlapping audio paths)
assert len(set(train_df["audio_path"]) & set(val_df["audio_path"])) == 0, "Leakage between train and val!"
assert len(set(train_df["audio_path"]) & set(test_df["audio_path"])) == 0, "Leakage between train and test!"
assert len(set(val_df["audio_path"]) & set(test_df["audio_path"])) == 0, "Leakage between val and test!"
print("[+] Split isolation verified: 0 overlapping audio paths across all splits.")


## 3. Audio Path, Sampling Rate, Transcript, and Language Validation
Verifies that audio paths resolve, checks sampling rate (16,000 Hz) and single-channel mono properties, and confirms valid UTF-8 transcripts with Spanish accented characters.

In [ ]:
def resolve_audio_path(rel_path: str, dataset_root: Path) -> Path:
    """Resolves manifest audio paths against the dataset root."""
    candidates = [
        dataset_root / rel_path,
        dataset_root / Path(rel_path).parts[-2] / Path(rel_path).name if len(Path(rel_path).parts) >= 2 else None,
        dataset_root / "audio" / Path(rel_path).parts[-2] / Path(rel_path).name if len(Path(rel_path).parts) >= 2 else None,
        dataset_root / "datasets" / "audio" / Path(rel_path).parts[-2] / Path(rel_path).name if len(Path(rel_path).parts) >= 2 else None,
        dataset_root / Path(rel_path).name,
        dataset_root / "audio" / Path(rel_path).name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return c
    raise FileNotFoundError(f"Audio clip not found: {rel_path} under root {dataset_root}")

# Verify sample English and Spanish audio files
en_sample = train_df[train_df["language"] == "en"].iloc[0]
es_sample = train_df[train_df["language"] == "es"].iloc[0]

en_p = resolve_audio_path(en_sample["audio_path"], DATASET_ROOT)
es_p = resolve_audio_path(es_sample["audio_path"], DATASET_ROOT)

en_wav, en_sr = sf.read(str(en_p), dtype="float32")
es_wav, es_sr = sf.read(str(es_p), dtype="float32")

print("[+] Audio Format Validation:")
print(f"    English sample: {en_p.name} | sr={en_sr} Hz | channels={en_wav.ndim} | duration={len(en_wav)/en_sr:.2f}s")
print(f"    Spanish sample: {es_p.name} | sr={es_sr} Hz | channels={es_wav.ndim} | duration={len(es_wav)/es_sr:.2f}s")

assert en_sr == 16000 and en_wav.ndim == 1, "English sample must be 16 kHz Mono!"
assert es_sr == 16000 and es_wav.ndim == 1, "Spanish sample must be 16 kHz Mono!"

# Verify UTF-8 transcripts and accents
print(f"\n[+] Transcript Verification:")
print(f"    EN: {en_sample['transcript'][:70]}...")
print(f"    ES: {es_sample['transcript'][:70]}...")


## 4. Multilingual Whisper Processor and Model Setup
Initializes the multilingual checkpoint `openai/whisper-base` (74M parameters) for ASR transcription (`task="transcribe"`).

**Key Principles**:
* Uses multilingual `openai/whisper-base`, **never** `whisper-base.en`, to support both English and Spanish.
* The training objective is strictly speech transcription (EN $\rightarrow$ EN, ES $\rightarrow$ ES).
* `forced_decoder_ids` is decoupled on both `model.config` and `model.generation_config` to prevent overriding Spanish speech with English prompt tokens.

In [ ]:
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration
)

MODEL_ID = "openai/whisper-base"
TARGET_TASK = "transcribe"  # Monolingual ASR only. Translation is out of scope.

print(f"[*] Loading Feature Extractor and Tokenizer for: {MODEL_ID}")
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, task=TARGET_TASK)
processor = WhisperProcessor.from_pretrained(MODEL_ID, task=TARGET_TASK)

en_prompts = tokenizer.get_decoder_prompt_ids(language="en", task=TARGET_TASK)
es_prompts = tokenizer.get_decoder_prompt_ids(language="es", task=TARGET_TASK)

print(f"[+] Decoder Start Token ID: {tokenizer.decoder_start_token_id} ('{tokenizer.decode([tokenizer.decoder_start_token_id])}')")
print(f"[+] English Prompt IDs:     {en_prompts}")
print(f"[+] Spanish Prompt IDs:     {es_prompts}")
print(f"[+] Pad Token ID:           {tokenizer.pad_token_id}")


## 5. Audio Preprocessing Pipeline and Data Collator
Defines `StoreListenerASRDataset` and `DataCollatorSpeechSeq2SeqWithPadding`:
* **30-second Window Policy**: Whisper operates on fixed 30.0s log-mel frames ($80 \times 3000$). Clips exceeding 30.0s (480,000 samples) are truncated; shorter clips are padded with silence during feature extraction.
* **Loss Masking**: Replaces target padding tokens with `-100` so that `CrossEntropyLoss` ignores padding.
* **Decoder Start Token**: Strips `decoder_start_token_id` (`50258`) from position 0 of labels so the forward pass right-shifts correctly.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
import torch
from torch.utils.data import Dataset

EXPECTED_SAMPLE_RATE = 16000
MAX_DURATION_SECONDS = 30.0
MAX_SAMPLES = int(EXPECTED_SAMPLE_RATE * MAX_DURATION_SECONDS)  # 480,000 samples

class StoreListenerASRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_root: Path, feature_extractor, tokenizer):
        self.df = df.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.feature_extractor = feature_extractor
        self.tokenizer = tokenizer
        self.prompt_cache = {
            "en": [tok_id for _, tok_id in self.tokenizer.get_decoder_prompt_ids(language="en", task="transcribe")],
            "es": [tok_id for _, tok_id in self.tokenizer.get_decoder_prompt_ids(language="es", task="transcribe")]
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        full_path = resolve_audio_path(row["audio_path"], self.dataset_root)
        waveform, sr = sf.read(str(full_path), dtype="float32")

        if sr != EXPECTED_SAMPLE_RATE:
            raise ValueError(f"Sample rate mismatch: {full_path.name} is {sr} Hz, expected {EXPECTED_SAMPLE_RATE} Hz.")
        if waveform.ndim > 1:
            waveform = waveform.squeeze()

        # Truncation policy for clips exceeding 30s
        if len(waveform) > MAX_SAMPLES:
            waveform = waveform[:MAX_SAMPLES]

        input_features = self.feature_extractor(waveform, sampling_rate=EXPECTED_SAMPLE_RATE).input_features[0]

        lang = str(row["language"]).strip().lower()
        transcript = str(row["transcript"]).strip()
        text_token_ids = self.tokenizer(transcript).input_ids
        labels = [self.tokenizer.decoder_start_token_id] + self.prompt_cache[lang] + text_token_ids

        return {
            "input_features": input_features,
            "labels": labels,
            "language": lang,
            "audio_path": row["audio_path"]
        }

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        batch["languages"] = [f["language"] for f in features]
        return batch

# Instantiate Datasets and Collator
train_dataset = StoreListenerASRDataset(train_df, DATASET_ROOT, feature_extractor, tokenizer)
val_dataset   = StoreListenerASRDataset(val_df, DATASET_ROOT, feature_extractor, tokenizer)
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=tokenizer.decoder_start_token_id)

# 2-sample sanity check (1 English, 1 Spanish)
sample_batch = data_collator([train_dataset[0], train_dataset[1]])
print("[+] Data Collator Sanity Check:")
print(f"    input_features shape: {sample_batch['input_features'].shape} (expected: [2, 80, 3000])")
print(f"    labels shape:         {sample_batch['labels'].shape}")
print(f"    -100 masked tokens:   {int((sample_batch['labels'] == -100).sum())}")
print(f"    batch languages:       {sample_batch['languages']}")


## 6. Parameter-Efficient Fine-Tuning (PEFT / LoRA) Configuration
Applies Low-Rank Adaptation (LoRA) to `q_proj` and `v_proj` attention projection layers of `openai/whisper-base` ($r=16, \alpha=32$). Enables gradient checkpointing and clears `forced_decoder_ids` on both `model.config` and `model.generation_config`.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

def build_lora_whisper_model(model_id: str, peft_config: LoraConfig):
    """
    Loads Whisper base model, enables gradient checkpointing, attaches LoRA adapters,
    and configures generation_config for unbiased bilingual ASR transcription.
    """
    model = WhisperForConditionalGeneration.from_pretrained(model_id, low_cpu_mem_usage=True)
    
    # 1. Base model.config
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    
    # 2. Generation config (used by model.generate() in Seq2SeqTrainer)
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.forced_decoder_ids = None
        model.generation_config.language = None       # Allow dynamic language auto-detection
        model.generation_config.task = "transcribe"   # Enforce ASR transcription, NOT translation
        model.generation_config.suppress_tokens = []
    
    # 3. Gradient checkpointing to save GPU VRAM
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    
    # 4. Wrap with LoRA
    lora_model = get_peft_model(model, peft_config)
    lora_model.print_trainable_parameters()
    return lora_model

print("[+] LoRA configuration defined for target modules: [q_proj, v_proj].")


## 7. Word Error Rate (WER) and Character Error Rate (CER) Evaluation
Computes both aggregate and language-specific metrics (`en_wer`, `en_cer`, `es_wer`, `es_cer`), evaluating normalized text (via Whisper's `BasicTextNormalizer`) alongside verbatim text.

In [ ]:
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
normalizer = BasicTextNormalizer()

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # Identify language per sample from label prefix tokens before replacing -100
    languages = []
    for l_seq in label_ids:
        valid_toks = [tok for tok in l_seq if tok != -100]
        if len(valid_toks) > 0:
            tok_str = tokenizer.decode([valid_toks[0]], skip_special_tokens=False)
            languages.append("es" if "es" in tok_str else "en")
        else:
            languages.append("en")

    # Replace -100 with pad_token_id for correct decoding
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)
    pred_ids  = np.where(pred_ids  != -100, pred_ids,  tokenizer.pad_token_id)

    pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    overall_wer_verb = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    overall_cer_verb = 100 * cer_metric.compute(predictions=pred_str, references=label_str)

    pred_str_norm  = [normalizer(p) for p in pred_str]
    label_str_norm = [normalizer(l) for l in label_str]

    valid_pairs = [(p, l) for p, l in zip(pred_str_norm, label_str_norm) if len(l.strip()) > 0]
    if valid_pairs:
        p_norm, l_norm = zip(*valid_pairs)
        overall_wer_norm = 100 * wer_metric.compute(predictions=list(p_norm), references=list(l_norm))
        overall_cer_norm = 100 * cer_metric.compute(predictions=list(p_norm), references=list(l_norm))
    else:
        overall_wer_norm, overall_cer_norm = 0.0, 0.0

    # Language-specific breakdowns
    en_preds_norm = [p for p, lang, l in zip(pred_str_norm, languages, label_str_norm) if lang == "en" and len(l.strip()) > 0]
    en_refs_norm  = [l for l, lang in zip(label_str_norm, languages) if lang == "en" and len(l.strip()) > 0]
    es_preds_norm = [p for p, lang, l in zip(pred_str_norm, languages, label_str_norm) if lang == "es" and len(l.strip()) > 0]
    es_refs_norm  = [l for l, lang in zip(label_str_norm, languages) if lang == "es" and len(l.strip()) > 0]

    en_wer = 100 * wer_metric.compute(predictions=en_preds_norm, references=en_refs_norm) if en_refs_norm else 0.0
    en_cer = 100 * cer_metric.compute(predictions=en_preds_norm, references=en_refs_norm) if en_refs_norm else 0.0
    es_wer = 100 * wer_metric.compute(predictions=es_preds_norm, references=es_refs_norm) if es_refs_norm else 0.0
    es_cer = 100 * cer_metric.compute(predictions=es_preds_norm, references=es_refs_norm) if es_refs_norm else 0.0

    return {
        "wer_normalized": round(overall_wer_norm, 2),
        "cer_normalized": round(overall_cer_norm, 2),
        "en_wer":         round(en_wer, 2),
        "en_cer":         round(en_cer, 2),
        "es_wer":         round(es_wer, 2),
        "es_cer":         round(es_cer, 2),
        "wer_verbatim":   round(overall_wer_verb, 2),
        "cer_verbatim":   round(overall_cer_verb, 2)
    }

print("[+] Metric computation module configured with WER, CER, and language-specific breakdowns.")


## 8. Training Arguments, Trainer Setup, and Checkpoint Export
Configures the Hugging Face `Seq2SeqTrainer` with conservative smoke-test parameters (50 steps, effective batch size 16), strict split isolation (only `train.csv` updates weights; `validation.csv` evaluated under `torch.no_grad()`; `test.csv` held out), and exports checkpoints to Google Drive or local output.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Output directory: Google Drive checkpoint folder if in Colab, otherwise local folder
if DATASET_ROOT == COLAB_DRIVE_ROOT:
    OUTPUT_DIR = COLAB_DRIVE_ROOT / "checkpoints" / "whisper-base-storelistener-lora"
else:
    OUTPUT_DIR = DATASET_ROOT / "whisper-base-storelistener-lora"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

eval_strat_key = "eval_strategy" if hasattr(Seq2SeqTrainingArguments, "eval_strategy") else "evaluation_strategy"
training_args_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 1e-4,
    "warmup_steps": 10,
    "max_steps": 50,                       # Smoke-test budget: 50 optimizer steps
    "gradient_checkpointing": True,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    eval_strat_key: "steps",
    "eval_steps": 25,
    "save_steps": 50,
    "logging_steps": 10,
    "report_to": ["none"],
    "predict_with_generate": True,
    "generation_max_length": 225,
    "remove_unused_columns": False,
    "dataloader_num_workers": 2 if sys.platform != "win32" else 0,
    "load_best_model_at_end": False
}

training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

def build_trainer(model, train_data, eval_data, collator, metrics_callback, args):
    return Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        data_collator=collator,
        compute_metrics=metrics_callback,
        tokenizer=processor.feature_extractor
    )

def export_checkpoints(model, processor, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(output_dir))
    processor.save_pretrained(str(output_dir))
    print(f"[+] Model adapter and processor exported to: {output_dir}")

print(f"[+] Configured Seq2SeqTrainer. Output destination: {OUTPUT_DIR}")


## 9. Spanish-to-English Speech Translation (Separate Phase 2 Task)
Whisper natively supports speech-to-text translation using `task="translate"` (`<|es|><|translate|>`).

> [!IMPORTANT]
> **Task Boundary**:
> * Phase 1 training focuses exclusively on **monolingual speech recognition** (EN $\rightarrow$ EN, ES $\rightarrow$ ES) to preserve acoustic fidelity.
> * Spanish-to-English translation is evaluated independently using Whisper's zero-shot multilingual capabilities and is **not** mixed into Phase 1 gradient updates.

In [ ]:
def evaluate_spanish_to_english_translation(model, processor, audio_path: Union[str, Path], device=None):
    """
    Evaluates Spanish audio -> English transcript using Whisper's translation task.
    Explicitly decouples translation evaluation from Phase 1 ASR training.
    """
    if device is None:
        device = getattr(model, "device", "cuda" if torch.cuda.is_available() else "cpu")
    wav, sr = sf.read(str(audio_path), dtype="float32")
    if sr != 16000:
        raise ValueError(f"Expected 16 kHz audio, found {sr} Hz.")
    if wav.ndim > 1:
        wav = wav.squeeze()
        
    inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    
    # Force Spanish source language and translation task tokens
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="es", task="translate")
    
    with torch.no_grad():
        predicted_ids = model.generate(inputs, forced_decoder_ids=forced_decoder_ids, max_new_tokens=128)
        
    translation_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return translation_text

print("[+] Standalone Spanish-to-English translation evaluation module defined.")


## 10. Training Safety Gate & Step-by-Step Colab Execution Guide

> [!CAUTION]
> **TRAINING SAFETY GUARD**: `EXECUTE_TRAINING = False` by default.  
> Running this notebook from top to bottom will verify configurations, test data collation, and check hardware **without** initiating training.
>
> ### Step-by-Step Google Colab Execution Procedure:
> 1. Open this notebook in Google Colab.
> 2. Select **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.
> 3. Execute Cells 1–3 to mount Google Drive and locate `/content/drive/MyDrive/whisper-finetune`.
> 4. Execute Cells 4–9 to install dependencies, verify audio samples, and build the collator.
> 5. To run the 50-step smoke training when authorized, set `EXECUTE_TRAINING = True` below.

In [ ]:
# =====================================================================
# EXPLICIT TRAINING OPT-IN FLAG
# Defaults to False. Set to True ONLY when authorized to train.
# =====================================================================
EXECUTE_TRAINING = False

print("=" * 70)
print("STORELISTENER TRAINING SAFETY GATE:")
print(f"  EXECUTE_TRAINING: {EXECUTE_TRAINING}")
print(f"  CUDA_READY:       {CUDA_READY}")
print(f"  DATASET_ROOT:     {DATASET_ROOT}")
print("=" * 70)

if not EXECUTE_TRAINING:
    print("\n[INFO] Training execution is DISABLED (EXECUTE_TRAINING = False).")
    print("       All pipeline components, manifests, and data collators verified successfully.")
    print("       No model weights were downloaded and no GPU compute was consumed.")
    print("       To authorize training in a future session: change EXECUTE_TRAINING = True.")
elif not CUDA_READY:
    raise RuntimeError(
        "BLOCKED: Training cannot run because CUDA is unavailable. "
        "Please switch your Colab runtime to a GPU: Runtime -> Change runtime type -> T4 GPU."
    )
else:
    print("\n[*] AUTHORIZED: Initializing LoRA model and launching training smoke-test...")
    lora_model = build_lora_whisper_model(MODEL_ID, lora_config)
    trainer = build_trainer(
        model=lora_model,
        train_data=train_dataset,
        eval_data=val_dataset,
        collator=data_collator,
        metrics_callback=compute_metrics,
        args=training_args
    )
    
    # Run 50-step smoke-test
    train_result = trainer.train()
    print("[+] Training completed successfully!")
    
    # Export LoRA adapter to Google Drive / local output
    export_checkpoints(lora_model, processor, OUTPUT_DIR)
    
    # Evaluate on validation split
    eval_metrics = trainer.evaluate()
    print(f"[+] Final Validation Metrics: {eval_metrics}")
